# 从 Attention 到 TTT：线性化之路（2/5）—— 分布视角：把 V 换成预测分布

上一册看清了 Softmax Attention 的原式：

$$o = \sum_{i=1}^{n} a_i v_i, \qquad a_i = \frac{e^{q \cdot k_i}}{Z}$$

这一册做文章里的**第一步推广**：不再把 $v_i$ 当成一个固定向量硬背下来，
而是把它看成从条件分布 $p(v \mid k_i)$ 里抽出来的样本。

## 1. 把加权求和改写成"期望"

原式可以理解为：对每个历史 token，取它的 $v_i$，再按 $a_i$ 加权。
推广之后，第 $i$ 项不再是固定的 $v_i$，而是分布 $p(v \mid k_i)$ 的期望：

$$o = \sum_{i=1}^{n} a_i \int p(v \mid k_i)\, v\, dv = \sum_{i=1}^{n} a_i \mathbb{E}_{p(v \mid k_i)}[v]$$

**为什么这算"推广"？** 如果取狄拉克分布（尖峰分布）

$$p(v \mid k_i) = \delta(v - v_i)$$

期望恰好就是 $v_i$，原式被完整恢复。也就是说：原式是这个分布形式的一个特例，
分布形式能表达更宽的一类函数。

## 2. 用条件正态分布替代"硬背"

狄拉克分布等价于把所有 $(k_i, v_i)$ 原样背下来——这正是我们想压缩掉的东西。
文章选了一个可计算的条件正态分布：

$$p(v \mid k_i) = \mathcal{N}\big(v;\, \mu_W(k_i),\, \sigma^2 I\big)$$

- $\mu_W(k_i)$：以 $k_i$ 为输入、参数为 $W$ 的**均值预测模型**；
- $\sigma$：预设的超参数（方差）。

正态分布的期望就是均值，于是 $\mathbb{E}[v \mid k_i] = \mu_W(k_i)$，输出变成：

$$o = \sum_{i=1}^{n} a_i \, \mu_W(k_i)$$

> 文章还提到：协方差也可以换成一般的预测模型，但最终只关心均值向量，
> 一般化协方差不会带来额外收益。

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
n, d = 4, 3

# 真实的历史 (k_i, v_i)
K = rng.standard_normal((n, d))
V = rng.standard_normal((n, d))

# 一个最简单的均值预测模型: mu(k) = W k  (线性回归, W 稍后在第 3 册学怎么训练)
W = rng.standard_normal((d, d)) * 0.3

def mu(k):
    return W @ k

# 文章式 (4): o = Σ a_i μ_W(k_i)，这里先随便给一组均匀权重演示"期望形式"
a = np.full(n, 1.0 / n)
o_dist = sum(a[i] * mu(K[i]) for i in range(n))

# 对照: 原始硬背形式 o = Σ a_i v_i
o_raw = a @ V

print("分布形式输出 μ_W(k) 混合:", o_dist.round(3))
print("硬背形式输出 v_i 混合:    ", o_raw.round(3))
print()
print("两者不同是预期的: μ_W 是学出来的预测, 不可能逐点还原硬背的 v_i")

## 3. 这一步省掉了什么、付出了什么

**收益**：只要 $\mu_W(k_i)$ 算得足够快，就不必把每个 $v_i$ 缓存下来，
**V Cache 被消掉，KV Cache 直接减半**（K 还留着）。

**代价**：$(\mu_W(k_i))$ 的拟合能力理论上不如硬配对的 $(k_i, v_i)$；
预测模型要靠后面的流式更新不断修正，才能跟上 token 之间的交互。

文章作者的判断是：这一步"半斤八两"——省了缓存，也让逐点记忆变成了泛化预测。

## 4. 小结（2/5）

- 把 $v_i$ 写成条件分布 $p(v\mid k_i)$ 的期望；狄拉克分布退化为原式，所以这是推广不是推翻；
- 选条件正态分布后，期望就是均值 $\mu_W(k_i)$，输出式变成 $o = \sum a_i \mu_W(k_i)$；
- 不再缓存 V，KV Cache 减半；K 和 $W$ 的训练问题留到下一册。

→ **下一册**：$\mu_W$ 的参数 $W$ 怎么估计？让历史 $(k_t, v_t)$ 依次到达，用 SGD 流式更新，
$W$ 本身就成了一个 RNN 的状态。